### Imports

In [20]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import datasets
import numpy as np
import os
import torch
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaForSequenceClassification,  Trainer, TrainingArguments, DataCollatorWithPadding

### Load and tokenize data

In [21]:
# model_name = 'distilroberta-base'
model_name = 'roberta-base'

In [22]:
tokenizer = RobertaTokenizer.from_pretrained(model_name)

In [23]:
dataset = 'values_labels'
# dataset = 'subclasess_labels'
# labels = ["Openness to change", "Self-enhancement", "Conservation", "Self-transcendence", "No Value"]
labels = [ "Self-direction: thought", "Self-direction: action", "Stimulation",  "Hedonism", "Achievement", "Power: dominance", "Power: resources", "Face", "Security: personal", "Security: societal", "Tradition", "Conformity: rules", "Conformity: interpersonal", "Humility", "Benevolence: caring", "Benevolence: dependability", "Universalism: concern", "Universalism: nature", "Universalism: tolerance", "No Value"]
num_labels = len(labels)

In [24]:
def load_dataset(directory, tokenizer, load_labels=True):
    sentences_file_path = os.path.join(directory, "sentences.tsv")
    labels_file_path = os.path.join(directory, f"{dataset}.tsv")
    
    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)
    encoded_sentences = tokenizer(data_frame["Text"].to_list(), truncation=True)

    if load_labels and os.path.isfile(labels_file_path):
        labels_frame = pd.read_csv(labels_file_path, encoding="utf-8", sep="\t", header=0)
        labels_frame = pd.merge(data_frame, labels_frame, on=["Text-ID", "Sentence-ID"], how="inner")
        labels_matrix = np.zeros((labels_frame.shape[0], len(labels)))
        for idx, label in enumerate(labels):
            if label in labels_frame.columns:
                labels_matrix[:, idx] = (labels_frame[label] >= 0.5).astype(int)
        encoded_sentences["labels"] = labels_matrix.tolist()

    encoded_sentences = datasets.Dataset.from_dict(encoded_sentences)
    
    return encoded_sentences, data_frame["Text-ID"].to_list(), data_frame["Sentence-ID"].to_list()

In [25]:
directory_test="datasets/valueeval24/test-english"
directory_train="datasets/valueeval24/training-english"
directory_validation="datasets/valueeval24/validation-english"

encoded_sentences_test, text_ids_test, sentence_ids_test = load_dataset(directory_test, tokenizer)
encoded_sentences_train, text_ids_train, sentence_ids_train = load_dataset(directory_train, tokenizer)
encoded_sentences_validation, text_ids_validation, sentence_ids_validation = load_dataset(directory_validation, tokenizer)

In [26]:
# Use a Subset of the Dataset
def create_subset(dataset, fraction=0.1):
    subset_size = int(len(dataset) * fraction)
    return dataset.select(range(subset_size))

subset_fraction = 0.5
encoded_sentences_train_subset = create_subset(encoded_sentences_train, subset_fraction)
encoded_sentences_test_subset = create_subset(encoded_sentences_test, subset_fraction)

### Loading model, defining loss and freeze weights

In [27]:
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)  

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# class CustomRobertaForSequenceClassification(RobertaForSequenceClassification):
#     def forward(self, input_ids=None, attention_mask=None, labels=None):
#         outputs = self.roberta(input_ids, attention_mask=attention_mask)
#         sequence_logits = outputs[0]  # Shape: [batch_size, sequence_length, num_labels]
#         cls_logits = sequence_logits[:, 0, :]  # Shape: [batch_size, num_labels]
#         logits = self.classifier(cls_logits)
#         if labels is not None:
#             # loss_fct = CustomLossWithAtLeastOnePositive(nn.BCEWithLogitsLoss())
#             loss_fct = nn.BCEWithLogitsLoss()
#             loss = loss_fct(logits, labels.float())
#             return (loss, logits)
#         return (logits,)

# model = CustomRobertaForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# # Freezing the weights
# # for param in model.roberta.parameters():
# #     param.requires_grad = False

# # Adding a final classification layer
# model.classifier = nn.Sequential(
#     nn.ReLU(),
#     nn.Dropout(0.3),
#     nn.Linear(model.config.hidden_size, num_labels),
# )

Some weights of CustomRobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Training parameters

In [28]:
def compute_metrics(pred):
    logits, labels = pred
    probs = torch.sigmoid(torch.tensor(logits))
    
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    best_threshold, best_f1 = 0, 0
    for t in thresholds:
        prediction = (probs >= t).int()
        _, _, f1, _ = precision_recall_fscore_support(labels, prediction, average='samples')
        if f1 > best_f1:
            best_threshold, best_f1 = t, f1
    # best_threshold = 0.5
    
    prediction = (probs >= best_threshold).int()
    max_probs = probs.argmax(dim=1)
    for i in range(prediction.shape[0]):
        if prediction[i].sum() == 0:  # No label selected
            prediction[i, max_probs[i]] = 1  # Assign the label with max probability

    
    # Calcul des métriques pour un problème multi-label
    precision, recall, f1, _ = precision_recall_fscore_support(labels, prediction, average='samples',zero_division=0)
    accuracy = accuracy_score(labels, prediction)
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [29]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [30]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,             
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    warmup_ratio=0.2
)

/homes/e21dabie/ProCom/roberta-env/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [31]:
optimizer = torch.optim.AdamW(
    model.parameters(),  # Parameters to optimize
    lr=2e-5,             # Learning rate
    weight_decay=0.01    # Weight decay for regularization
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_sentences_train_subset,
    eval_dataset=encoded_sentences_test_subset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None)
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.135000,0.130655,0.376304,0.546710,0.507665,0.629874
2,0.118000,0.123520,0.368067,0.548002,0.509306,0.637024
3,0.096400,0.122843,0.446458,0.580412,0.553325,0.644426


/homes/e21dabie/ProCom/roberta-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/homes/e21dabie/ProCom/roberta-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/homes/e21dabie/ProCom/roberta-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitaliz

In [ ]:
predictions_output = trainer.predict(encoded_sentences_validation)
logits = predictions_output.predictions
probs = torch.sigmoid(torch.tensor(logits))
thresh = 0.5
binary_predictions = (probs >= thresh).int()
for i in range(probs.size(0)):  # Iterate over each example in the batch
    if binary_predictions[i].sum() == 0:  # No label predicted
        max_idx = torch.argmax(probs[i])  # Index of the max probability
        binary_predictions[i, max_idx] = 1
        
print("Final binary predictions:")
print(binary_predictions)
# for i, sentence in enumerate(encoded_sentences_validation["input_ids"]):
#     predicted_labels = [
#         label for label, pred in zip(labels, binary_predictions[i].tolist()) if pred == 1
#     ]
#     print(f"Sentence {i+1}: {tokenizer.decode(sentence)}")
#     print(f"Predicted labels: {predicted_labels}")
#     print()

Final binary predictions:
tensor([[0, 0, 1, 0],
        [0, 0, 1, 0],
        [0, 0, 1, 0],
        ...,
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]], dtype=torch.int32)


In [ ]:
sentence = "Your sentence here."

# Tokenize the input
inputs = tokenizer(sentence, truncation=True, padding=True, return_tensors="pt")

# Make prediction
with torch.no_grad():
    outputs = base_model(**inputs)
    logits = outputs[0]
    probs = torch.sigmoid(logits).squeeze().numpy() 

threshold = 0.5
binary_predictions = (probs >= threshold).astype(int)
predicted_labels = [label for label, pred in zip(labels, binary_predictions) if pred == 1]
print("Predicted labels:", predicted_labels)

In [14]:
test_metrics = trainer.evaluate(encoded_sentences_validation)
print("Évaluation sur le jeu de test:", test_metrics)

/homes/e21dabie/ProCom/roberta-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Évaluation sur le jeu de test: {'eval_loss': 0.1626603603363037, 'eval_accuracy': 0.48731884057971014, 'eval_f1': 0.49292595264817485, 'eval_precision': 0.49175500984075865, 'eval_recall': 0.49618111468956877, 'eval_runtime': 51.3067, 'eval_samples_per_second': 290.488, 'eval_steps_per_second': 36.311, 'epoch': 4.0}
